# 🛒 SmartPantry AI — Complete Build (Sprint 1 + Sprint 2)
**Portfolio Project | Agentic AI + Advanced Analytics**

### One notebook. One session. Zero tokens.

This combines Data Foundation (Sprint 1) and Analytics Engine (Sprint 2) into a single notebook.
Run all cells top to bottom — everything happens in one Colab session, so there's no
"file not found" issue between sprints.

### How to use this notebook
1. Open it directly from GitHub: `colab.research.google.com/github/YOUR-USERNAME/SmartPantry-AI/blob/main/SmartPantry_Complete.ipynb`
2. In **Cell 1**, change `GITHUB_USER` to your GitHub username (edit directly in Colab, not on GitHub)
3. **Runtime → Run all**
4. Everything runs: data generation → forecasting → order optimisation → menu coverage → spend analytics → dashboards
5. When done: **File → Save a copy in GitHub** → select your repo → commit message → OK

That last step is how your work gets saved back to GitHub — built into Colab's menu, no token needed.

### What you get at the end
| Output | What it is |
|---|---|
| 6 CSV/JSON datasets | Pantry, menu, history, forecast, orders, spend |
| 2 dashboards | Sprint 1 pantry overview + Sprint 2 analytics dashboard |
| Full forecast + order list | Ready to act on this week |

### Session restart workflow (every time Colab disconnects)
1. Reopen the same Colab URL (from GitHub)
2. Runtime → Run all
3. Takes about 1-2 minutes total — all data regenerates fresh, nothing is lost
4. Optional: File → Save a copy in GitHub, to snapshot this run's outputs

---

# PART 1 — Sprint 1: Data Foundation
---

## Cell 1 — GitHub Repo Setup
Run this first every session. Clones your repo so all data paths are consistent.

In [ ]:
# ── CELL 1: Clone GitHub repo into Colab session ─────────────────────────────
import os, subprocess

# ── UPDATE THESE TWO LINES ────────────────────────────────────────────────────
GITHUB_USER = 'your-github-username'   # e.g. 'karthika'
GITHUB_REPO = 'SmartPantry-AI'
# ─────────────────────────────────────────────────────────────────────────────

REPO_URL   = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
REPO_DIR   = f'/content/{GITHUB_REPO}'
DATA_DIR   = f'{REPO_DIR}/data'
OUTPUT_DIR = f'{REPO_DIR}/outputs'

if os.path.exists(REPO_DIR):
    print('Repo already cloned this session. Pulling latest...')
    subprocess.run('git pull', shell=True, cwd=REPO_DIR)
else:
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(f'git clone {REPO_URL} {REPO_DIR}', shell=True)

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'\n✓ Ready')
print(f'  Repo dir   : {REPO_DIR}')
print(f'  Data dir   : {DATA_DIR}')
print(f'  Output dir : {OUTPUT_DIR}')
print(f'  Files now  : {os.listdir(DATA_DIR)}')

## Cell 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import random
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)
today = datetime.today().date()

print(f'Imports done | Today: {today}')

## Cell 3 — Family Profile
> **Customise this** — update to match your actual family before running.

In [ ]:
family_profile = {
    'family_id': 'HOME_001',
    'name': 'My Home',
    'members': [
        {'name': 'Adult 1',  'age_group': 'adult',  'diet': 'vegetarian'},
        {'name': 'Adult 2',  'age_group': 'adult',  'diet': 'non-vegetarian'},
        {'name': 'Child 1',  'age_group': 'child',  'diet': 'vegetarian'},
        {'name': 'Child 2',  'age_group': 'child',  'diet': 'vegetarian'},
    ],
    'total_members': 4,
    'adults': 2,
    'children': 2,
    'monthly_grocery_budget_inr': 12000,
    'preferred_order_day': 'Sunday',
    'order_platform': 'Zepto/Blinkit',
    'city': 'Bengaluru'
}

with open(f'{DATA_DIR}/family_profile.json', 'w') as f:
    json.dump(family_profile, f, indent=2)

print(f'Family : {family_profile["total_members"]} members | Budget ₹{family_profile["monthly_grocery_budget_inr"]:,}/mo')
print(f'Saved  → {DATA_DIR}/family_profile.json')

## Cell 4 — Item Master (40 items, 13 categories)

In [ ]:
items = [
    # (name, category, unit, monthly_4pax, reorder_threshold, lead_days, price/unit, shelf_days, perishable, pattern)
    ('Rice (Basmati)',      'Grains',      'kg',      5.0,  1.5, 1, 120, 365, False, 'steady'),
    ('Wheat Atta',         'Grains',      'kg',      4.0,  1.0, 1,  55, 180, False, 'steady'),
    ('Poha',               'Grains',      'kg',      0.5,  0.2, 1,  50, 180, False, 'steady'),
    ('Oats',               'Grains',      'kg',      0.5,  0.2, 1,  80, 180, False, 'steady'),
    ('Pasta',              'Grains',      'kg',      0.5,  0.2, 1,  80, 365, False, 'occasional'),
    ('Toor Dal',           'Pulses',      'kg',      1.5,  0.4, 1,  95, 365, False, 'steady'),
    ('Chana Dal',          'Pulses',      'kg',      0.5,  0.2, 1,  90, 365, False, 'steady'),
    ('Moong Dal',          'Pulses',      'kg',      0.5,  0.2, 1,  90, 365, False, 'steady'),
    ('Rajma',              'Pulses',      'kg',      0.3,  0.1, 1,  85, 365, False, 'occasional'),
    ('Milk',               'Dairy',       'litre',  15.0,  3.0, 0,  64,   2, True,  'daily'),
    ('Curd',               'Dairy',       'kg',      2.0,  0.5, 0,  40,   7, True,  'steady'),
    ('Paneer',             'Dairy',       'kg',      0.5,  0.2, 0, 320,   7, True,  'occasional'),
    ('Butter',             'Dairy',       'gm',    200.0, 50.0, 1, 350,  30, True,  'steady'),
    ('Eggs',               'Protein',     'pcs',    24.0,  6.0, 0,   8,  21, True,  'steady'),
    ('Chicken',            'Protein',     'kg',      1.5,  0.5, 0, 180,   2, True,  'occasional'),
    ('Onion',              'Vegetables',  'kg',      3.0,  0.8, 0,  30,  30, False, 'steady'),
    ('Tomato',             'Vegetables',  'kg',      2.0,  0.5, 0,  40,   7, True,  'steady'),
    ('Potato',             'Vegetables',  'kg',      2.0,  0.5, 0,  25,  30, False, 'steady'),
    ('Garlic',             'Vegetables',  'gm',    200.0, 50.0, 0,  40,  30, False, 'steady'),
    ('Ginger',             'Vegetables',  'gm',    100.0, 30.0, 0,  20,  14, True,  'steady'),
    ('Spinach',            'Vegetables',  'bundle',  2.0,  1.0, 0,  30,   3, True,  'weekly'),
    ('Capsicum',           'Vegetables',  'pcs',     4.0,  1.0, 0,  20,   7, True,  'weekly'),
    ('Sunflower Oil',      'Oils & Fats', 'litre',   2.0,  0.5, 1, 140, 365, False, 'steady'),
    ('Mustard Oil',        'Oils & Fats', 'litre',   0.5,  0.2, 1, 130, 365, False, 'occasional'),
    ('Ghee',               'Oils & Fats', 'gm',    250.0, 80.0, 1,   1, 365, False, 'steady'),
    ('Salt',               'Spices',      'kg',      0.5,  0.2, 2,  20, 730, False, 'slow'),
    ('Turmeric',           'Spices',      'gm',    100.0, 30.0, 2,   0, 365, False, 'slow'),
    ('Red Chilli Powder',  'Spices',      'gm',    100.0, 30.0, 2,   0, 365, False, 'slow'),
    ('Coriander Powder',   'Spices',      'gm',    100.0, 30.0, 2,   0, 365, False, 'slow'),
    ('Cumin Seeds',        'Spices',      'gm',    100.0, 30.0, 2,   0, 365, False, 'slow'),
    ('Garam Masala',       'Spices',      'gm',     50.0, 15.0, 2,   0, 365, False, 'slow'),
    ('Tea',                'Beverages',   'gm',    500.0,150.0, 1,   1, 365, False, 'steady'),
    ('Coffee',             'Beverages',   'gm',    200.0, 60.0, 1,   2, 365, False, 'steady'),
    ('Sugar',              'Sweeteners',  'kg',      1.5,  0.4, 1,  42, 365, False, 'steady'),
    ('Jaggery',            'Sweeteners',  'gm',    200.0, 80.0, 1,   0, 180, False, 'occasional'),
    ('Bread',              'Bakery',      'loaf',    2.0,  1.0, 0,  45,   7, True,  'weekly'),
    ('Biscuits',           'Snacks',      'pack',    3.0,  1.0, 1,  35, 180, False, 'steady'),
    ('Maggi Noodles',      'Snacks',      'pack',    4.0,  1.0, 1,  14, 365, False, 'occasional'),
    ('Coconut Milk',       'Canned',      'tin',     2.0,  0.5, 2,  55, 730, False, 'occasional'),
    ('Tomato Ketchup',     'Condiments',  'gm',    500.0,100.0, 1,   0, 365, False, 'slow'),
]

# Unit-normalized prices (per gm for gm-unit items)
price_corrections = {
    'Ghee': 550/500, 'Tea': 200/250, 'Coffee': 450/200,
    'Turmeric': 14/100, 'Red Chilli Powder': 20/100,
    'Coriander Powder': 15/100, 'Cumin Seeds': 25/100,
    'Garam Masala': 45/50, 'Butter': 350/500,
    'Tomato Ketchup': 80/500, 'Garlic': 40/200,
    'Ginger': 20/100, 'Jaggery': 30/200,
}

cols = ['item_name','category','unit','monthly_consumption_4pax','reorder_threshold',
        'lead_time_days','price_inr_per_unit','shelf_life_days','is_perishable','consumption_pattern']
df_master = pd.DataFrame(items, columns=cols)
df_master['item_id'] = ['ITEM_' + str(i+1).zfill(3) for i in range(len(df_master))]
df_master['daily_consumption_rate'] = (df_master['monthly_consumption_4pax'] / 30).round(4)
df_master['price_inr_per_unit'] = df_master.apply(
    lambda r: price_corrections.get(r['item_name'], r['price_inr_per_unit']), axis=1
)
df_master.to_csv(f'{DATA_DIR}/item_master.csv', index=False)

print(f'Item master: {len(df_master)} items | {df_master["category"].nunique()} categories')
print(df_master.groupby('category')['item_name'].count().sort_values(ascending=False).to_string())
print(f'\nSaved → {DATA_DIR}/item_master.csv')

## Cell 5 — Pantry Snapshot

In [ ]:
def simulate_current_stock(row):
    max_stock = row['monthly_consumption_4pax'] * 1.5
    fractions = {'daily':(.05,.25),'weekly':(.1,.5),'steady':(.15,.85),'occasional':(.2,1.0),'slow':(.4,1.2)}
    lo, hi = fractions.get(row['consumption_pattern'], (.3,.7))
    return round(max_stock * np.random.uniform(lo, hi), 2)

df_pantry = df_master.copy()
df_pantry['current_stock'] = df_pantry.apply(simulate_current_stock, axis=1)
df_pantry['last_restocked_date'] = [
    str(today - timedelta(days=random.randint(0, 20))) for _ in range(len(df_pantry))
]
df_pantry['days_to_empty'] = (
    df_pantry['current_stock'] / df_pantry['daily_consumption_rate']
).clip(upper=60).round(1)
df_pantry['stock_status'] = pd.cut(
    df_pantry['days_to_empty'], bins=[-1,3,7,14,60],
    labels=['CRITICAL','LOW','MODERATE','SUFFICIENT']
)
df_pantry['reorder_needed']   = df_pantry['current_stock'] <= df_pantry['reorder_threshold']
df_pantry['snapshot_date']    = str(today)
df_pantry['monthly_spend_inr'] = (df_pantry['monthly_consumption_4pax'] * df_pantry['price_inr_per_unit']).round(0)
df_pantry.to_csv(f'{DATA_DIR}/pantry_snapshot.csv', index=False)

sc = df_pantry['stock_status'].value_counts()
print('Pantry Snapshot:')
for s, e in [('CRITICAL','🔴'),('LOW','🟠'),('MODERATE','🟡'),('SUFFICIENT','🟢')]:
    print(f'  {e} {s}: {sc.get(s,0)} items')
print(f'\n  Reorder needed: {df_pantry["reorder_needed"].sum()} items')
print(f'  Est. monthly spend: ₹{df_pantry["monthly_spend_inr"].sum():,.0f}')
print(f'\nSaved → {DATA_DIR}/pantry_snapshot.csv')

## Cell 6 — Weekly Menu Plan

In [ ]:
menu_plan = [
    {'day':'Monday',    'meal':'Breakfast','dish':'Poha',
     'ingredients':{'Poha':.2,'Onion':.1,'Mustard Oil':.02,'Cumin Seeds':.005,'Salt':.005,'Turmeric':.002}},
    {'day':'Monday',    'meal':'Lunch',    'dish':'Dal Tadka + Rice',
     'ingredients':{'Toor Dal':.15,'Rice (Basmati)':.3,'Onion':.1,'Tomato':.1,'Ghee':.02,'Cumin Seeds':.005,'Turmeric':.003,'Salt':.005}},
    {'day':'Monday',    'meal':'Dinner',   'dish':'Roti + Paneer Curry',
     'ingredients':{'Wheat Atta':.3,'Paneer':.2,'Tomato':.15,'Onion':.1,'Sunflower Oil':.03,'Garam Masala':.005,'Salt':.005}},
    {'day':'Tuesday',   'meal':'Breakfast','dish':'Oats Porridge',
     'ingredients':{'Oats':.15,'Milk':.3,'Sugar':.03,'Jaggery':.02}},
    {'day':'Tuesday',   'meal':'Lunch',    'dish':'Rajma Chawal',
     'ingredients':{'Rajma':.15,'Rice (Basmati)':.3,'Onion':.15,'Tomato':.15,'Sunflower Oil':.03,'Cumin Seeds':.005,'Garam Masala':.005}},
    {'day':'Tuesday',   'meal':'Dinner',   'dish':'Roti + Egg Curry',
     'ingredients':{'Wheat Atta':.3,'Eggs':2.,'Onion':.1,'Tomato':.1,'Sunflower Oil':.03,'Red Chilli Powder':.005,'Salt':.005}},
    {'day':'Wednesday', 'meal':'Breakfast','dish':'Bread Butter + Eggs',
     'ingredients':{'Bread':.5,'Butter':.03,'Eggs':2.}},
    {'day':'Wednesday', 'meal':'Lunch',    'dish':'Palak Dal + Rice',
     'ingredients':{'Moong Dal':.12,'Spinach':.3,'Rice (Basmati)':.3,'Ghee':.02,'Garlic':.02,'Cumin Seeds':.005,'Turmeric':.003}},
    {'day':'Wednesday', 'meal':'Dinner',   'dish':'Chicken Curry + Roti',
     'ingredients':{'Chicken':.5,'Wheat Atta':.3,'Onion':.2,'Tomato':.15,'Ginger':.02,'Garlic':.02,'Sunflower Oil':.04,'Garam Masala':.008,'Salt':.005}},
    {'day':'Thursday',  'meal':'Breakfast','dish':'Poha + Curd',
     'ingredients':{'Poha':.2,'Curd':.1,'Salt':.003}},
    {'day':'Thursday',  'meal':'Lunch',    'dish':'Chana Dal + Rice',
     'ingredients':{'Chana Dal':.15,'Rice (Basmati)':.3,'Tomato':.1,'Onion':.1,'Sunflower Oil':.025,'Coriander Powder':.005,'Turmeric':.003}},
    {'day':'Thursday',  'meal':'Dinner',   'dish':'Pasta Arrabiata',
     'ingredients':{'Pasta':.25,'Tomato':.2,'Capsicum':1.,'Garlic':.015,'Sunflower Oil':.03,'Red Chilli Powder':.005,'Salt':.005}},
    {'day':'Friday',    'meal':'Breakfast','dish':'Oats + Milk',
     'ingredients':{'Oats':.15,'Milk':.3,'Sugar':.02}},
    {'day':'Friday',    'meal':'Lunch',    'dish':'Aloo Sabzi + Roti',
     'ingredients':{'Potato':.3,'Wheat Atta':.3,'Onion':.1,'Turmeric':.003,'Cumin Seeds':.005,'Salt':.005,'Sunflower Oil':.025}},
    {'day':'Friday',    'meal':'Dinner',   'dish':'Maggi + Egg',
     'ingredients':{'Maggi Noodles':2.,'Eggs':2.,'Onion':.05,'Tomato Ketchup':.03}},
    {'day':'Saturday',  'meal':'Breakfast','dish':'Poori + Aloo',
     'ingredients':{'Wheat Atta':.4,'Potato':.2,'Sunflower Oil':.1,'Salt':.005,'Cumin Seeds':.003}},
    {'day':'Saturday',  'meal':'Lunch',    'dish':'Chicken Biryani',
     'ingredients':{'Rice (Basmati)':.5,'Chicken':.5,'Onion':.3,'Tomato':.2,'Ghee':.04,'Ginger':.02,'Garlic':.02,'Garam Masala':.01,'Salt':.005}},
    {'day':'Saturday',  'meal':'Dinner',   'dish':'Paneer Tikka + Roti',
     'ingredients':{'Paneer':.25,'Wheat Atta':.3,'Capsicum':1.,'Curd':.1,'Red Chilli Powder':.005,'Salt':.003}},
    {'day':'Sunday',    'meal':'Breakfast','dish':'Bread Omelette',
     'ingredients':{'Bread':.5,'Eggs':4.,'Onion':.1,'Butter':.02,'Salt':.003}},
    {'day':'Sunday',    'meal':'Lunch',    'dish':'Dal Makhani + Rice',
     'ingredients':{'Rajma':.1,'Toor Dal':.1,'Rice (Basmati)':.3,'Butter':.03,'Milk':.1,'Onion':.1,'Tomato':.15,'Garam Masala':.008,'Salt':.005}},
    {'day':'Sunday',    'meal':'Dinner',   'dish':'Chole + Poori',
     'ingredients':{'Chana Dal':.2,'Wheat Atta':.4,'Onion':.15,'Tomato':.15,'Sunflower Oil':.08,'Coriander Powder':.008,'Cumin Seeds':.005,'Salt':.005}},
]

df_menu = pd.DataFrame(menu_plan)
df_menu.to_csv(f'{DATA_DIR}/weekly_menu.csv', index=False)

demand_agg = {}
for row in menu_plan:
    for item, qty in row['ingredients'].items():
        demand_agg[item] = demand_agg.get(item, 0) + qty

df_demand = pd.DataFrame([
    {'item_name': k, 'weekly_menu_demand': round(v, 4)}
    for k, v in demand_agg.items()
]).sort_values('weekly_menu_demand', ascending=False).reset_index(drop=True)
df_demand.to_csv(f'{DATA_DIR}/menu_demand.csv', index=False)

print(f'Menu: {len(df_menu)} meals | {len(df_demand)} unique ingredients')
print(f'Saved → {DATA_DIR}/weekly_menu.csv')
print(f'Saved → {DATA_DIR}/menu_demand.csv')

## Cell 7 — Consumption History (8 weeks)

In [ ]:
history_rows = []
for _, item in df_master.iterrows():
    base_weekly = item['monthly_consumption_4pax'] / 4.33
    for w in range(8):
        week_start = today - timedelta(weeks=w+1)
        actual = max(0, round(base_weekly * np.random.normal(1.0, 0.12), 3))
        history_rows.append({
            'item_id': item['item_id'], 'item_name': item['item_name'],
            'week_start': str(week_start), 'consumed_qty': actual,
            'unit': item['unit'], 'family_members': 4
        })

df_history = pd.DataFrame(history_rows)
df_history.to_csv(f'{DATA_DIR}/consumption_history.csv', index=False)

print(f'History: {len(df_history)} rows | {df_history["item_name"].nunique()} items | {df_history["week_start"].nunique()} weeks')
print(f'Saved → {DATA_DIR}/consumption_history.csv')

## Cell 8 — Visualise & Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SmartPantry AI — Sprint 1: Pantry Overview', fontsize=14, fontweight='bold')

# Chart 1: Stock status
ax1 = axes[0]
status_order  = ['CRITICAL','LOW','MODERATE','SUFFICIENT']
status_colors = ['#E53935','#FB8C00','#FDD835','#43A047']
sc2 = df_pantry['stock_status'].value_counts().reindex(status_order, fill_value=0)
bars = ax1.bar(sc2.index, sc2.values, color=status_colors, edgecolor='white', width=0.6)
for bar in bars:
    h = bar.get_height()
    if h > 0: ax1.text(bar.get_x()+bar.get_width()/2, h+.3, str(int(h)), ha='center', fontweight='bold')
ax1.set_title('Stock Status', fontweight='bold')
ax1.set_ylabel('Items')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Chart 2: Days to empty
ax2 = axes[1]
urgent = df_pantry.nsmallest(15,'days_to_empty')[['item_name','days_to_empty','stock_status']].copy()
cmap = {'CRITICAL':'#E53935','LOW':'#FB8C00','MODERATE':'#FDD835','SUFFICIENT':'#43A047'}
ax2.barh(urgent['item_name'], urgent['days_to_empty'], color=urgent['stock_status'].map(cmap), edgecolor='white')
ax2.axvline(x=3, color='#E53935', linestyle='--', linewidth=1, alpha=.7)
ax2.axvline(x=7, color='#FB8C00', linestyle='--', linewidth=1, alpha=.7)
ax2.set_title('Days to Empty — 15 Most Urgent', fontweight='bold')
ax2.set_xlabel('Days')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# Chart 3: Spend by category
ax3 = axes[2]
spend = df_pantry.groupby('category')['monthly_spend_inr'].sum().sort_values()
bars3 = ax3.barh(spend.index, spend.values, color='#5C6BC0', edgecolor='white', alpha=.85)
for bar in bars3:
    w = bar.get_width()
    ax3.text(w+30, bar.get_y()+bar.get_height()/2, f'₹{int(w):,}', va='center', fontsize=8)
ax3.set_title('Monthly Spend by Category (₹)', fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sprint1_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print('=' * 55)
print('  SmartPantry AI — Sprint 1 COMPLETE ✓')
print('=' * 55)
files = ['item_master.csv','pantry_snapshot.csv','weekly_menu.csv',
         'menu_demand.csv','consumption_history.csv','family_profile.json']
for fname in files:
    path = f'{DATA_DIR}/{fname}'
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'  ✓ {fname:<35} {size:,} bytes')
print(f'  ✓ outputs/sprint1_overview.png')
print()
print('Next: Open Sprint2_Analytics_Engine.ipynb in Colab')

# PART 2 — Sprint 2: Analytics Engine
---

> Uses `df_master`, `df_pantry`, `df_history`, `df_demand`, `profile` directly from Part 1 — already in memory, no file reload needed.

## Analytics Setup
Reuses `DATA_DIR`, `OUTPUT_DIR`, and all dataframes already created in Part 1 above — no need to reload from disk.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# df_master, df_pantry, df_history, df_demand, profile, DATA_DIR, OUTPUT_DIR
# are already defined from Part 1 — just confirming they exist:
print('Using data already in memory from Part 1:')
print(f'  df_master  : {len(df_master)} items')
print(f'  df_pantry  : {len(df_pantry)} items')
print(f'  df_history : {len(df_history)} rows')
print(f'  df_demand  : {len(df_demand)} ingredients')
print(f'  profile    : {profile["total_members"]} members, ₹{profile["monthly_grocery_budget_inr"]:,}/mo budget')

## Cell 3 — Module 1: Depletion Forecast
**FMCG mapping:** velocity + promo uplift → days of supply + replenishment priority index

In [ ]:
def compute_depletion_forecast(df_pantry, df_history, df_demand):
    df_hist = df_history.copy()
    df_hist['week_start'] = pd.to_datetime(df_hist['week_start'])

    def weighted_avg(grp):
        grp = grp.sort_values('week_start')
        n = len(grp)
        weights = np.array([1]*max(0,n-4) + [2]*min(4,n), dtype=float)[-n:]
        return np.average(grp['consumed_qty'].values, weights=weights)

    hist_rates = (
        df_hist.groupby('item_name')
        .apply(weighted_avg, include_groups=False)
        .reset_index(name='weighted_weekly_rate')
    )
    cv_df = (
        df_hist.groupby('item_name')['consumed_qty']
        .agg(['mean','std'])
        .assign(cv=lambda x: (x['std']/x['mean']*100).round(1))
        .reset_index()[['item_name','cv']]
    )

    df_fc = df_pantry[['item_id','item_name','category','unit','current_stock',
                        'daily_consumption_rate','reorder_threshold','lead_time_days',
                        'price_inr_per_unit','stock_status','reorder_needed',
                        'is_perishable','shelf_life_days']].copy()
    df_fc = df_fc.merge(hist_rates, on='item_name', how='left')
    df_fc = df_fc.merge(cv_df,      on='item_name', how='left')
    df_fc = df_fc.merge(df_demand,  on='item_name', how='left')

    df_fc['weighted_weekly_rate'] = df_fc['weighted_weekly_rate'].fillna(df_fc['daily_consumption_rate']*7)
    df_fc['weekly_menu_demand']   = df_fc['weekly_menu_demand'].fillna(0)
    df_fc['cv']                   = df_fc['cv'].fillna(10)

    df_fc['hist_daily_rate']       = (df_fc['weighted_weekly_rate']/7).round(4)
    df_fc['menu_daily_demand']     = (df_fc['weekly_menu_demand']/7).round(4)
    df_fc['combined_daily_demand'] = df_fc[['hist_daily_rate','menu_daily_demand']].max(axis=1).round(4)
    df_fc['menu_uplift_pct']       = np.where(
        df_fc['hist_daily_rate']>0,
        ((df_fc['menu_daily_demand']-df_fc['hist_daily_rate'])/df_fc['hist_daily_rate']*100).round(1), 0
    )
    df_fc['forecast_days_to_empty'] = np.where(
        df_fc['combined_daily_demand']>0,
        (df_fc['current_stock']/df_fc['combined_daily_demand']).clip(0,60).round(1), 60.0
    )
    df_fc['urgency_score'] = (
        (1 - df_fc['forecast_days_to_empty'].clip(0,14)/14)*60 +
        (df_fc['cv'].clip(0,30)/30)*20 +
        (df_fc['menu_uplift_pct'].clip(0,100)/100)*20
    ).round(1).clip(0,100)
    df_fc['must_order_by'] = df_fc.apply(
        lambda r: str(today + timedelta(days=max(0, r['forecast_days_to_empty']-r['lead_time_days']-1))), axis=1
    )
    df_fc['forecast_status'] = pd.cut(
        df_fc['forecast_days_to_empty'], bins=[-1,3,7,14,60],
        labels=['CRITICAL','LOW','MODERATE','OK']
    )
    return df_fc.sort_values('urgency_score', ascending=False).reset_index(drop=True)

df_forecast = compute_depletion_forecast(df_pantry, df_history, df_demand)
df_forecast.to_csv(f'{DATA_DIR}/forecast_output.csv', index=False)

sc = df_forecast['forecast_status'].value_counts()
print('Depletion Forecast ✓')
for s,e in [('CRITICAL','🔴'),('LOW','🟠'),('MODERATE','🟡'),('OK','🟢')]:
    print(f'  {e} {s}: {sc.get(s,0)} items')
print(f'  Menu uplift >20%: {(df_forecast["menu_uplift_pct"]>20).sum()} items')
print()
cols = ['item_name','current_stock','unit','forecast_days_to_empty','menu_uplift_pct','urgency_score','must_order_by']
print('Top 8 by urgency:')
print(df_forecast[cols].head(8).to_string(index=False))
print(f'\nSaved → {DATA_DIR}/forecast_output.csv')

## Cell 4 — Module 2: Order Optimisation

In [ ]:
def build_order_list(df_forecast, monthly_budget_inr, horizon_days=7):
    weekly_budget = monthly_budget_inr / 4
    df = df_forecast.copy()
    df['qty_to_order'] = np.maximum(0, (df['combined_daily_demand']*14) - df['current_stock']).round(2)
    df['order_flag']   = 'SKIP'
    df.loc[df['reorder_needed'] | (df['forecast_days_to_empty']<=horizon_days), 'order_flag'] = 'MUST ORDER'
    df.loc[(df['order_flag']=='SKIP') & (df['urgency_score']>50) & (df['forecast_days_to_empty']<=14), 'order_flag'] = 'RECOMMENDED'
    df['order_cost_inr'] = (df['qty_to_order'] * df['price_inr_per_unit']).round(0)

    candidates = df[df['order_flag'].isin(['MUST ORDER','RECOMMENDED'])].copy()
    candidates = candidates.sort_values(['order_flag','urgency_score'], ascending=[True,False])
    cumulative, selected = 0, []
    for _, row in candidates.iterrows():
        cost = row['order_cost_inr']
        if row['order_flag']=='MUST ORDER':
            selected.append(True); cumulative += cost
        elif cumulative + cost <= weekly_budget:
            selected.append(True); cumulative += cost
        else:
            selected.append(False)
    candidates['within_budget'] = selected
    return candidates[candidates['within_budget']], cumulative, weekly_budget

df_order, total_cost, weekly_budget = build_order_list(df_forecast, profile['monthly_grocery_budget_inr'])
df_order.to_csv(f'{DATA_DIR}/order_list.csv', index=False)

must = (df_order['order_flag']=='MUST ORDER').sum()
rec  = (df_order['order_flag']=='RECOMMENDED').sum()
print(f'Order List ✓  |  Weekly budget: ₹{weekly_budget:,.0f}')
print(f'  Must order  : {must} items')
print(f'  Recommended : {rec} items')
print(f'  Total cost  : ₹{total_cost:,.0f}  ({total_cost/weekly_budget*100:.0f}% of budget)')
print(f'  Remaining   : ₹{weekly_budget-total_cost:,.0f}')
print()
print(df_order[['item_name','unit','qty_to_order','order_cost_inr','order_flag','urgency_score']].to_string(index=False))
print(f'\nSaved → {DATA_DIR}/order_list.csv')

## Cell 5 — Module 3: Menu Coverage Check

In [ ]:
def check_menu_coverage(df_forecast, df_demand):
    df = df_demand.merge(
        df_forecast[['item_name','current_stock','unit','forecast_days_to_empty','urgency_score']],
        on='item_name', how='left'
    )
    df['stock_covers_menu'] = df['current_stock'] >= df['weekly_menu_demand']
    df['shortfall']         = (df['weekly_menu_demand'] - df['current_stock']).clip(lower=0).round(3)
    df['coverage_pct']      = (df['current_stock']/df['weekly_menu_demand']*100).clip(0,100).round(1)
    df['risk_level']        = pd.cut(df['coverage_pct'], bins=[-1,50,80,100,101],
                                     labels=['HIGH RISK','MEDIUM RISK','LOW RISK','COVERED'])
    return df.sort_values('coverage_pct').reset_index(drop=True)

df_coverage = check_menu_coverage(df_forecast, df_demand)
df_coverage.to_csv(f'{DATA_DIR}/menu_coverage.csv', index=False)

covered   = df_coverage['stock_covers_menu'].sum()
uncovered = (~df_coverage['stock_covers_menu']).sum()
print(f'Menu Coverage ✓  |  {covered}/{len(df_coverage)} ingredients covered')
if uncovered > 0:
    print(f'  ⚠️  {uncovered} gaps — meals at risk:')
    print(df_coverage[~df_coverage['stock_covers_menu']][
        ['item_name','unit','weekly_menu_demand','current_stock','shortfall','coverage_pct']
    ].to_string(index=False))
else:
    print('  ✅ All ingredients covered!')
print(f'\nSaved → {DATA_DIR}/menu_coverage.csv')

## Cell 6 — Module 4: Spend Analytics

In [ ]:
def compute_spend_analytics(df_fc, profile):
    df_sp = df_fc.copy()
    df_sp['monthly_spend_inr'] = (df_sp['combined_daily_demand']*30*df_sp['price_inr_per_unit']).round(0)
    by_cat = df_sp.groupby('category')['monthly_spend_inr'].sum().sort_values(ascending=False)
    total  = by_cat.sum()
    budget = profile['monthly_grocery_budget_inr']
    top5   = df_sp.nlargest(5,'monthly_spend_inr')[['item_name','category','monthly_spend_inr']]
    return {
        'total_monthly_est': round(total,0),
        'budget': budget,
        'utilisation_pct': round(total/budget*100,1),
        'budget_headroom': round(budget-total,0),
        'by_category': by_cat.round(0).to_dict(),
        'by_category_pct': (by_cat/total*100).round(1).to_dict(),
        'top5_items': top5.to_dict('records')
    }

spend = compute_spend_analytics(df_forecast, profile)
with open(f'{DATA_DIR}/spend_analytics.json','w') as f:
    json.dump(spend, f, indent=2)

print(f'Spend Analytics ✓')
print(f'  Monthly est  : ₹{spend["total_monthly_est"]:,.0f}')
print(f'  Budget       : ₹{spend["budget"]:,}')
print(f'  Utilisation  : {spend["utilisation_pct"]}%')
print(f'  Headroom     : ₹{spend["budget_headroom"]:,.0f}')
print()
for cat, amt in spend['by_category'].items():
    pct = spend['by_category_pct'][cat]
    print(f'  {cat:<18} ₹{amt:>6,.0f}  ({pct:.0f}%)')
print(f'\nSaved → {DATA_DIR}/spend_analytics.json')

## Cell 7 — Dashboard (4 charts)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16,12))
fig.suptitle('SmartPantry AI — Sprint 2: Analytics Dashboard', fontsize=15, fontweight='bold')
SC = {'CRITICAL':'#E53935','LOW':'#FB8C00','MODERATE':'#FDD835','OK':'#43A047'}

# Chart 1: Urgency scores
ax1 = axes[0,0]
top12 = df_forecast.head(12).copy()
bcolors = top12['forecast_status'].map(SC).fillna('#90A4AE')
bars = ax1.barh(top12['item_name'][::-1], top12['urgency_score'][::-1], color=bcolors[::-1], edgecolor='white', height=.7)
ax1.axvline(x=70, color='#E53935', linestyle='--', alpha=.5, linewidth=1)
ax1.axvline(x=50, color='#FB8C00', linestyle='--', alpha=.5, linewidth=1)
for bar in bars:
    w = bar.get_width()
    ax1.text(w+1, bar.get_y()+bar.get_height()/2, f'{w:.0f}', va='center', fontsize=9)
ax1.set_xlabel('Urgency Score (0–100)'); ax1.set_title('Top 12 by Urgency Score', fontweight='bold')
ax1.legend(handles=[mpatches.Patch(color=c,label=s) for s,c in SC.items()], fontsize=8, loc='lower right')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Chart 2: Scatter urgency vs days
ax2 = axes[0,1]
for status, color in SC.items():
    mask = df_forecast['forecast_status']==status
    ax2.scatter(df_forecast.loc[mask,'urgency_score'], df_forecast.loc[mask,'forecast_days_to_empty'],
                c=color, label=status, alpha=.8, s=70, edgecolors='white', linewidths=.5)
for _, row in df_forecast.head(5).iterrows():
    ax2.annotate(row['item_name'], (row['urgency_score'],row['forecast_days_to_empty']), fontsize=7, xytext=(5,3), textcoords='offset points')
ax2.axhline(y=7, color='#FB8C00', linestyle='--', alpha=.5, linewidth=1)
ax2.axhline(y=3, color='#E53935', linestyle='--', alpha=.5, linewidth=1)
ax2.set_xlabel('Urgency Score'); ax2.set_ylabel('Days to Empty')
ax2.set_title('Urgency vs Days to Empty', fontweight='bold'); ax2.legend(fontsize=8)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# Chart 3: Spend by category
ax3 = axes[1,0]
cats = list(spend['by_category'].keys()); vals = list(spend['by_category'].values())
pcts = list(spend['by_category_pct'].values())
palette = plt.cm.Set2(np.linspace(0,1,len(cats)))
bars3 = ax3.barh(cats[::-1], vals[::-1], color=palette[::-1], edgecolor='white', height=.7)
for bar, pct in zip(bars3, pcts[::-1]):
    w = bar.get_width()
    ax3.text(w+20, bar.get_y()+bar.get_height()/2, f'₹{int(w):,} ({pct:.0f}%)', va='center', fontsize=8)
ax3.set_title(f'Monthly Spend by Category | Total ₹{spend["total_monthly_est"]:,.0f}', fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# Chart 4: Menu coverage
ax4 = axes[1,1]
cov20 = df_coverage.head(20).copy()
rmap  = {'HIGH RISK':'#E53935','MEDIUM RISK':'#FB8C00','LOW RISK':'#FDD835','COVERED':'#43A047'}
ccolors = cov20['risk_level'].map(rmap).fillna('#90A4AE')
ax4.barh(cov20['item_name'][::-1], cov20['coverage_pct'][::-1], color=ccolors[::-1], edgecolor='white', height=.7)
ax4.axvline(x=100, color='#43A047', linestyle='--', alpha=.6, linewidth=1.5)
ax4.set_xlabel('Coverage %'); ax4.set_title('Menu Coverage — 20 Most Constrained', fontweight='bold')
ax4.legend(handles=[mpatches.Patch(color=c,label=l) for l,c in rmap.items()], fontsize=8)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sprint2_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {OUTPUT_DIR}/sprint2_dashboard.png')

## Cell 8 — Save to GitHub
File → Save a copy in GitHub, OR run this cell to commit from Colab.

In [ ]:
import subprocess
from datetime import datetime

def push_to_github(commit_message=None):
    if not commit_message:
        commit_message = f'Sprint 2 outputs: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
    cmds = ['git add -A', f'git commit -m "{commit_message}"', 'git push']
    for cmd in cmds:
        result = subprocess.run(cmd, shell=True, cwd=REPO_DIR, capture_output=True, text=True)
        status = '✓' if result.returncode==0 or 'nothing to commit' in result.stdout else '⚠️'
        print(f'{status} {cmd}')
        if result.returncode != 0 and 'nothing to commit' not in result.stdout:
            print(f'   {result.stderr.strip()}')

# ── Option A: Use Colab's built-in GitHub save (recommended — no token needed)
# File → Save a copy in GitHub → select your repo → commit message → OK
print('RECOMMENDED: File → Save a copy in GitHub')
print()

# ── Option B: push via git (needs GITHUB_TOKEN in Colab Secrets)
# Uncomment below if you prefer terminal-style push
# push_to_github('Sprint 2: Analytics engine complete')

print('=' * 60)
print('  SmartPantry AI — Sprint 2 COMPLETE ✓')
print('=' * 60)
print()
print(f'  Forecast : {len(df_forecast)} items scored')
print(f'  Orders   : {len(df_order)} items | ₹{total_cost:,.0f}')
print(f'  Coverage : {covered}/{len(df_coverage)} menu ingredients covered')
print(f'  Spend    : ₹{spend["total_monthly_est"]:,.0f}/mo ({spend["utilisation_pct"]}% of budget)')
print()
print('  Ready for Sprint 3: Claude API Agent Brain')
print(f'  Repo: https://github.com/{GITHUB_USER}/{GITHUB_REPO}')

In [ ]:
print('=' * 65)
print('  SmartPantry AI — FULL BUILD COMPLETE (Sprint 1 + Sprint 2) ✓')
print('=' * 65)
print()
print('All datasets created in:', DATA_DIR)
print('All charts saved in    :', OUTPUT_DIR)
print()
print('Files on disk:')
import os
for fname in sorted(os.listdir(DATA_DIR)):
    print(f'  ✓ data/{fname}')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  ✓ outputs/{fname}')
print()
print('NEXT STEP: File → Save a copy in GitHub')
print('  This commits this notebook (with all cell outputs) back to your repo.')
print('  Your charts and printed results will be visible directly on GitHub.')
print()
print('Note: the CSV/JSON/PNG files themselves live in the Colab session only')
print('(temporary disk) unless you separately upload them to GitHub via')
print('Add file -> Upload files in your repo. The notebook + its outputs')
print('(including printed tables and charts) ARE saved via Save a copy in GitHub.')
print()
print('Ready for Sprint 3: Claude API Agent Brain — just ask when ready!')